In [1]:
from ccdproc import ImageFileCollection, subtract_overscan, trim_image, subtract_bias, flat_correct, Combiner
import re, os, sys, glob
from datetime import datetime
import numpy as np
from astropy.io import fits
from astropy.nddata import CCDData
from astropy.stats import mad_std
from functools import partial
from multiprocessing import Pool
from multiprocessing.shared_memory import SharedMemory


In [2]:
def image_extensions(image, is_hdu=False):
    
    if is_hdu:
        image_indices = np.arange(len(image))
        selection = [hdu.size > 0 for hdu in image]

    else: 
        with fits.open(image) as hdus:
            image_indices = np.arange(len(hdus))
            selection = [hdu.size > 0 for hdu in hdus]
    
    return image_indices[selection]

#================================================================================

def ifc_filters(ifc, 
                obstype_selection='*',
                filter_keywords='filter1,filter2', 
                ignore_values='open,no_filter,unavailable'):

    
    keywds = filter_keywords.split(',')
    igvalues = '|'.join(ignore_values.split(','))

    objects_ifc = ifc.filter(obstype=obstype_selection, regex_match=True)

    #.Obtendo uma lista com todos os possiveis filtros presentes no FileCollection
    filters = objects_ifc.values(keywds[0],unique=True)
    for i in np.arange(1,len(keywds)):
        filters.extend(objects_ifc.values(keywds[i],unique=True))

    #.Removendo itens duplicados nas duas (ou mais) rodas de filtros
    filters=list(set(filters))
    filters.sort()

    #..removendo filtros "abertos" (segundo o input dado em 'ignore_values')
    p = re.compile(igvalues,re.IGNORECASE)
    flag = [p.search(flt) == None for flt in filters]
    filters = [filt for filt,flg in zip(filters,flag) if flg]

    return filters

#================================================================================

def image_filters(file_list, 
                  filter_keywords='filter1,filter2',
                  ignore_values='open,no_filter,unavailable'):
    
    filter_list=[]
    for file in file_list:

        hdr=fits.getheader(file, ext=0)
        keywds = filter_keywords.split(',')
        filters=[hdr[key] for key in keywds]

        igvals = '|'.join(ignore_values.split(','))
        p = re.compile(igvals,re.IGNORECASE)
        flag = [p.search(flt) == None for flt in filters]
        filters = [filters for filters,flg in zip(filters,flag) if flg]
        filter_list.append(filters[0])
    
    return filter_list

#================================================================================

def image_to_ccddata(image, is_hdu=False):

    ccddata=[]
    
    if is_hdu: hdul = image
    else: hdul = fits.open(image)

    for hdu in hdul:
        if hdu.size == 0: data = ()
        else: data = hdu.data
        ccddata.append(CCDData(data, meta=hdu.header, unit="adu"))

    if not is_hdu: hdul.close()

    return ccddata

#=============================================================================

def center_mode(image_matrix, frac_size=0.5):

    #  Orientation
    ysz, xsz = image_matrix.shape
    yc, xc = int(ysz/2), int(xsz/2)                     # Center
    dy, dx = int(ysz*frac_size/2), int(xsz*frac_size/2) # Step

    #  Limits
    a, b = yc - dy, yc + dy
    c, d = xc - dx, xc + dy

    roi = image_matrix[a:b, c:d]
    med, mad = np.median(roi), mad_std(roi)
    hist, bin_edge = np.histogram(roi,bins='rice',range=(med-6*mad,med+6*mad))
    bin_cen = (bin_edge[:-1]+bin_edge[1:])/2

    return bin_cen[np.argmax(hist)]

#=============================================================================

def flat_scale(flat_list, normalize=True):
    """
    Given a list of flat field fits files, parse through all extensions, 
    recording the median inside a selectable region near the center of the 
    image.

    Arguments
    ---------
        flat_list : list
            list of flat field images to analyse
        normalize : bool
            when set, a normalization across all amplifiers is carried out
            and the returned table will contain numbers between 0 and 1.

    Returns
    -------
        scales : numpy 2D array (n_files, n_extensions)
            median of the central region values for every extension in each 
            image of the input file list. If "normalize" parameter is set 
            then this table is divided by its maximum value.
    """
    
    for j,flat in enumerate(flat_list,start=0):
        hdus = fits.open(flat)
        if (j == 0):
            image_indices = np.arange(len(hdus))
            selection = [hdu.size > 0 for hdu in hdus]
            image_indices = image_indices[selection]
            scales = np.zeros([len(flat_list),len(image_indices)])
    
        for i,idx in enumerate(image_indices,start=0):
            scales[j,i] = center_mode(hdus[idx].data, frac_size=0.90)

        hdus.close()
    
    scales = np.amax(scales, axis=1)
    
    if not normalize:
        scales /= scales[0]
        
    return scales

#=============================================================================

def find_filter(image_file, filter_list, filter_keywords):
    """
    Given an image and some filter keywords, search if any of these keywords
    have a filter matching the filter list. The function will return the index
    within the filter list where the match occurred.

    Arguments
    ---------
        image_file : string or list of strings
            filename of the image
        filter_list : list of strings
            possible filter names to match with the filter keywords
        filter_keywords : list of strings
            header keywords contining filter information

    Returns
    -------
        Index within the filter list, where the filter match occurred
    """

    if type(image_file) is not list: image_file = [ image_file ]

    keywds = filter_keywords.split(',')
    filt_index=[]

    for image in image_file:

        hdr=fits.getheader(image, ext=0)
        for key in keywds:
            filt = hdr[key]
            if filt in filter_list:
                filt_index.append(filter_list.index(filt))
        
    return filt_index

#=============================================================================

def iraf2python(my_string):
    
    """
    This function is from Bruno Quint.
    
    Parse a string containing [XX:XX, YY:YY] to pixels.
    Parameter
    ---------
        my_string : str
    """
    
    my_string = my_string.strip()
    my_string = my_string.replace('[', '')
    my_string = my_string.replace(']', '')
    x, y = my_string.split(',')
    x = x.split(':')
    y = y.split(':')

    x = np.array(x, dtype=int)
    y = np.array(y, dtype=int)

    x[0] -= 1
    y[0] -= 1

    return x, y

#=============================================================================

def image_to_memory(input, name='shm'):

    if isinstance(input,str): hdul = fits.open(input)
    else: hdul = input

    image_exts = image_extensions(hdul, is_hdu=True)
    hdrs = np.array([hdul[ext].header.tostring() for ext in image_exts])
    data = np.stack([hdul[ext].data for ext in image_exts])
    
    shm_hdrs = SharedMemory(create=True, size=hdrs.nbytes, name=name+'_hdrs')
    shm_data = SharedMemory(create=True, size=data.nbytes, name=name+'_data')

    shm_hdrs_array = np.ndarray(hdrs.shape, hdrs.dtype, buffer=shm_hdrs.buf)
    shm_data_array = np.ndarray(data.shape, data.dtype, buffer=shm_data.buf)

    np.copyto(shm_hdrs_array, hdrs)
    np.copyto(shm_data_array, data)

    out = [{'buffer': name+'_hdrs', 'shape': hdrs.shape, 'dtype': hdrs.dtype}]
    out.append({'buffer': name+'_data', 'shape': data.shape, 'dtype': data.dtype})

    shm_hdrs.close()
    shm_data.close()

    if not hdul[0]._has_data: 
        hdr0 = np.array(hdul[0].header.tostring())
        shm_hdr0 = SharedMemory(create=True, size=hdr0.nbytes, name=name+'_hdr0')
        shm_hdr0_array = np.ndarray(hdr0.shape, hdr0.dtype, buffer=shm_hdr0.buf)
        np.copyto(shm_hdr0_array, hdr0)
        out.append({'buffer': name+'_hdr0', 'shape': hdr0.shape, 'dtype': hdr0.dtype})
        shm_hdr0.close()

    if isinstance(input,str): hdul.close()

    return out

#=============================================================================

def memory_to_ccddata(shm_dict):

    if len(shm_dict) > 2: 
        shm_hdr0 = SharedMemory(name=shm_dict[2]['buffer'])
        hdr0 = np.ndarray(shm_dict[2]['shape'], shm_dict[2]['dtype'], buffer=shm_hdr0.buf)
        ccddata = [(CCDData((), meta=fits.Header.fromstring(str(hdr0)), unit="adu"))]
        shm_hdr0.close()
    else: ccddata=[]
    
    shm_hdrs = SharedMemory(name=shm_dict[0]['buffer'])
    shm_data = SharedMemory(name=shm_dict[1]['buffer'])
    hdrs = np.ndarray(shm_dict[0]['shape'], shm_dict[0]['dtype'], buffer=shm_hdrs.buf)
    data = np.ndarray(shm_dict[1]['shape'], shm_dict[1]['dtype'], buffer=shm_data.buf)

    for i in range(shm_dict[0]['shape'][0]):
        ccddata.append(CCDData(data[i], 
                               meta=fits.Header.fromstring(str(hdrs[i])), 
                               unit='adu'))

    shm_hdrs.close()
    shm_data.close()

    return ccddata

#=============================================================================

def unlink_memory(shm_dict):

    for mem_dict in shm_dict:

        mem_block = SharedMemory(name=mem_dict['buffer']) 
        mem_block.close()
        mem_block.unlink()


In [3]:
def reduce_image(image_file, 
                 master_bias=None, 
                 master_flat=None,
                 shared_memory=False,
                 merge_amplifiers=True):

    #.Abrindo o arquivo fits da imagem
    hdul = fits.open(image_file)
    image_exts = image_extensions(hdul, is_hdu=True)

    get_date = datetime.now().strftime("%b %d %Y %H:%M")
    fstr = os.path.basename(image_file)
    
    #..preparando dados do master bias
    if master_bias is not None:
        if shared_memory:
            mbias_ccd = memory_to_ccddata(master_bias)
            mbias_buf = [SharedMemory(name=master_bias[i]['buffer']) for i in range(len(master_bias))]
        else: mbias_ccd = master_bias
        bstr = os.path.basename(mbias_ccd[0].header['FILENAME'])

    #.preparando dados dos master flats
    if master_flat is not None:
        if shared_memory: 
            mflat_ccd = memory_to_ccddata(master_flat)
            mflat_buf = [SharedMemory(name=master_flat[i]['buffer']) for i in range(len(master_flat))]
        else: mflat_ccd = master_flat
        flstr = os.path.basename(mflat_ccd[0].header['FILENAME'])

    #.preparando para juntar os amplificadores
    can_merge = (len(image_exts) > 1) and merge_amplifiers and (hdul[0].header['OBSTYPE']=='OBJECT')
    if can_merge:
         #..coletando binagem e dimensoes da imagem
        ccdsum = hdul[image_exts[0]].header['CCDSUM']
        bin_x, bin_y = int(ccdsum[0]), int(ccdsum[2])
        imsz_x, imsz_y = iraf2python(hdul[image_exts[0]].header['DETSIZE'])
        imsz_x, imsz_y = int(imsz_x[1]/bin_x), int(imsz_y[1]/bin_y)
        #..SOI gap
        inst = hdul[0].header['INSTRUME']
        if inst.find('SOI') >= 0: 
            xgap = np.array([0,0,102/bin_x,102/bin_x],dtype=int)
            ygap = np.array([0,0,0,0], dtype=int)
        else: xgap, ygap = np.full(4,0), np.full(4,0)

        #..inicializando imagem final
        img_merge = np.full((imsz_y+np.amax(ygap), imsz_x+np.amax(xgap)), np.nan, dtype=np.float32)

    #.Loop ao longo das extensoes (amplificadores) da imagem
    proc_string = f".Processing {fstr:1s}"
    skip = np.full(len(image_exts),True)
    for ne,ext in enumerate(image_exts):

        #.Lendo dados desta extensao
        proc_string+=f" [{ext:1.0f}]"
        img = CCDData(hdul[ext].data, meta=hdul[ext].header, unit="adu")
        
        #.Realizando correcao de OVERSCAN
        if ('BIASSEC' in img.header):
            proc_string+="o"
            biassec = img.header['BIASSEC']        
            img = subtract_overscan(img, 
                        fits_section=biassec, 
                        model=None, median=True,
                        add_keyword={'overscan': f"{get_date} Overscan is {biassec}; model=median"})

            #.SOI exception
            #.(keyword TRIMSEC errada no header)
            trimsec = img.header['TRIMSEC']
            if (hdul[0].header['INSTRUME'].find('SOI') >= 0):
                if (ext in [1,3]): trimsec = '[29:540,1:2048]'
                else: trimsec = '[28:539,1:2048]'

            img = trim_image(img, 
                        fits_section=trimsec,
                        add_keyword={'trim': f"{get_date} Trim is {trimsec}"})
            
            #.Atualizando o header
            imsz = img.shape
            img.header['DATASEC'] = f"[1:{imsz[1]},1:{imsz[0]}]"
            del img.header['BIASSEC']
            del img.header['TRIMSEC']
            del img.header['BZERO']
            del img.header['BSCALE']

            skip[ne] = False
        else: 
            proc_string+="-"


        #.Realizando correcao de BIAS
        if ('ZEROCOR' not in img.header) and (master_bias is not None): 
            proc_string+="z"
            img = subtract_bias(img, 
                        mbias_ccd[ext],
                        add_keyword={'ZEROCOR': f"{get_date} Zero is {bstr}[{ext}]"})

            skip[ne] = False
        else:
            proc_string+="-"

        #.Realizando correcao de FLAT
        if ('FLATCOR' not in img.header) and (master_flat is not None):
            proc_string+="f"
            fscl = mflat_ccd[ext].header['FLATNORM']
            img = flat_correct(img, 
                        mflat_ccd[ext], 
                        norm_value=1,
            add_keyword={'FLATCOR': f"{get_date} Flat is {flstr}[{ext}] (norm={fscl:.1f})"})

            skip[ne] = False
        else:
            proc_string+="-"

        #.Juntando amplificadores
        if can_merge:
            proc_string+="m"
            #..coletando dimensoes e posicao relativa deste amplificador
            ampos_x, ampos_y = iraf2python(img.header['DETSEC'])
            ampos_x = (np.array(ampos_x)/bin_x).astype(int) + xgap[ne]
            ampos_y = (np.array(ampos_y)/bin_y).astype(int) + ygap[ne]
            
            #..escrevendo dados deste amplificador no objeto CCDDdata da imagem final
            img_merge[ampos_y[0]:ampos_y[1],ampos_x[0]:ampos_x[1]] = img.data
        else:
            proc_string+="-"

        #.Se alguma operacao foi realizada sobre a imagem, atualizar dados e cabecalho no HDU
        if not skip[ne]: 
            hdul[ext].data = img.data.astype(np.float32)
            hdul[ext].header = img.header
    

    #.Salvando a imagem processada 
    if can_merge:
        
        #..preparando o header da imagem combinada
        if image_exts[0] != 0: 
            hdr = hdul[0].header
            hdr.extend(hdul[1].header, unique=True)
        else: hdr = hdul[1].header
        #..removendo keywords que nao serao mais necessarias
        keystodel = ['DATASEC', 'CCDSEC', 'AMPSEC', 'DETSEC', 'NEXTEND', 'EXTNAME']
        for keyw in keystodel: del hdr[keyw]
        #..ajustando keywords para refeletir o novo estado da imagem
        hdr['DETSIZE']=f"[1:{imsz_x},1:{imsz_y}]"
        hdr['CCDSIZE']=f"[1:{imsz_x},1:{imsz_y}]"
        hdr.insert('NAXIS', ('NAXIS1', imsz_x, "Axis length"), after=True)
        hdr.insert('NAXIS1', ('NAXIS2', imsz_y, "Axis length"), after=True)   
        hdr.append(('AMPMERGE', f"{get_date} Merged {len(image_exts)} amps"))

        #..construindo o objeto CCDData para armazenar a imagem final
        img = CCDData(img_merge, meta=hdr, unit='adu')
        #..escrevendo para o arquivo (overwrite)
        img.write(image_file,hdu_mask=None,hdu_uncertainty=None,overwrite=True)
        
    else:
        if np.sum(skip) is not len(image_exts): 
            hdul.writeto(image_file, overwrite=True)
        hdul.close()
    print(proc_string, flush=True)


def reduce_image_mp(image_file, flat_idx, 
                    master_bias, 
                    master_flats_list,
                    merge_amplifiers=True):
    
    master_flat = master_flats_list[flat_idx]

    #.executando a reducao da imagem
    reduce_image(image_file, 
                  master_bias = master_bias,
                  master_flat = master_flat,
                  shared_memory = True,
                  merge_amplifiers = merge_amplifiers)




In [4]:
def process_bias(ifc, 
                 combined_bias='master_bias.fits',
                 delete_bias=True,
                 multiprocessing=True):
    
    #.separando imagens de BIAS em uma lista 
    bias_ifc = ifc.filter(obstype='BIAS|ZERO', regex_match=True)
    file_list = bias_ifc.files
    
    #.corrigindo OVERSCAN nas imagens de BIAS
    if multiprocessing: 
        with Pool() as pool:
            pool.map(partial(reduce_image, master_bias=None, master_flat=None,
                             merge_amplifiers=False), file_list)
    else: 
        for file in file_list: 
            reduce_image(file)

    #.combinando imagens de BIAS em um 'master_bias'
    combine_bias(file_list, delete_images=delete_bias, output=combined_bias, 
                 multiprocessing=multiprocessing)

    ifc.refresh()
    

def combine_bias(image_list, 
                 delete_images=True, 
                 output='master_bias.fits',
                 multiprocessing=True):
    
    #.tratando excessoes
    nbias = len(image_list)
    if (not os.path.isfile(output)):
        if (nbias > 0): 
            print(f".Creating '{os.path.basename(output)}': {nbias} images")
        else: sys.exit(".ABORTING 'master bias' creation: no BIAS images found")
    else: 
        print(f".Using '{os.path.basename(output)}' image found in directory")
        return

    #.criando imagem de saida (a partir da 1a da lista)
    mbias = fits.open(image_list[0])

    #.obtendo extensoes com imagem
    image_exts = image_extensions(mbias, is_hdu=True)

    #.combinando bias (por extensao)
    if multiprocessing:
        with Pool() as pool:
            result = pool.map(partial(combine_bias_extension, bias_list=image_list), 
                              image_exts)
    else:
        result=[]
        for ext in image_exts:
            result.append(combine_bias_extension(ext, bias_list=image_list))
    
    #.colocando as extensoes combinadas na imagem de saida 
    for res in result:
        ext = res[0]
        mbias[ext].data = res[1].data
        mbias[ext].header = res[1].header

    #.salvando imagem de saida
    mbias[0].header['FILENAME']=output
    mbias.writeto(output, overwrite=True)
    mbias.close()

    if delete_images: 
        for file in image_list: 
            try: os.remove(file)
            except OSError as e: print("Error: %s - %s." % (e.filename, e.strerror))


def combine_bias_extension(ext, bias_list):

    #..agrupando as imagens de bias deste amplificador
    ccd_list = [ CCDData.read(bias_list[j], hdu=ext, unit="adu")
                 for j in np.arange(len(bias_list)) ]

    #..preparando objeto combinador
    comb = Combiner(ccd_list, dtype=np.float32)
    #..processando opcoes do combinador
    comb.sigma_clipping(low_thresh=3, high_thresh=3, func='median', dev_func='mad_std')
    #..realizando combinacao
    comb_bias = comb.average_combine()

    #..pegando o cabecalho da primeira imagem e atualizando
    comb_bias.header = fits.getheader(bias_list[0], ext=ext)
    for n,imgn in enumerate(bias_list, start=1): 
        fstr = os.path.basename(imgn)
        comb_bias.header.append((f"IMCMB{n:03}", f"{fstr}[{ext}]"))
    comb_bias.header.append(('NCOMBINE', len(ccd_list), '# images combined'))

    #..retornando imagem processada desta extensao
    return (ext, comb_bias)



In [5]:
def process_flat(ifc, 
                 filter_keywords='filter1,filter2', 
                 master_bias='master_bias.fits',
                 combined_flats='master_flat.fits',
                 delete_flats=True,
                 multiprocessing=True):

    combined_flats = combined_flats.split('.fits')[0]
  
    #.separando imagens de FLAT em uma lista 
    flat_ifc = ifc.filter(obstype='FLAT|SFLAT|DFLAT', regex_match=True)
    file_list = flat_ifc.files

    #.aplicando correcao de OVERSCAN e BIAS nas imagens de FLAT
    if multiprocessing: 
        #..carregando 'master bias' em um buffer de memoria
        mbias_ccd = image_to_memory(master_bias)
        #..executando a reducao com multiprocessamento
        with Pool() as pool:
            pool.map(partial(reduce_image, master_bias=mbias_ccd, 
                             master_flat=None, shared_memory=True,
                             merge_amplifiers=False), file_list)
        #..liberando buffers de memoria 
        unlink_memory(mbias_ccd)
    else: 
        #..carregando imagem 'master bias' em um objeto CCDDATA
        mbias_ccd = image_to_ccddata(master_bias)   
        #..executando a reducao imagem por imagem
        for file in file_list: 
            reduce_image(file, master_bias=mbias_ccd, 
                         master_flat=None, merge_amplifiers=False)

    #.combinando FLATS por filtro
    filters = ifc_filters(flat_ifc, filter_keywords=filter_keywords, 
                          obstype_selection='*')
    keywds = filter_keywords.split(',')

    for fn,filt in enumerate(filters, start=1):

        #.separando lista de imagens de FLAT neste filtro
        filt_list = list(flat_ifc.files_filtered(**{keywds[0]: filt}))
        for i in np.arange(1,len(keywds)): 
            filt_list.append(list(flat_ifc.files_filtered(**{keywds[i]: filt})))
        filt_list = list(filter(None, filt_list))

        print(f"Creating '{os.path.splitext(os.path.basename(combined_flats))[0]}"+
            f"{fn}.fits': {len(filt_list)} images ({filt})")

        #.combinando imagens deste filtro em um 'master flat'
        combine_flat(filt_list, delete_images=delete_flats, 
                     output=f"{combined_flats}{fn}.fits", 
                     multiprocessing=multiprocessing)

    #.atualizando ImageFileCollection
    ifc.refresh()


def combine_flat(image_list, 
                 delete_images=True, 
                 output='master_flat.fits', 
                 multiprocessing=True):

    #.criando imagem de saida (a partir da 1a da lista)
    mflat = fits.open(image_list[0])

    #.obtendo extensoes com imagem
    image_exts = image_extensions(mflat, is_hdu=True)

    #.gerando fatores de escala entre os flats da lista
    fscales = flat_scale(image_list, normalize=True)

    #.combinando bias (por extensao)
    if multiprocessing:
        with Pool() as pool:
            result = pool.map(partial(combine_flat_extension, flat_list=image_list, 
                                      scaling=fscales), image_exts)
    else:
        result=[]
        for ext in image_exts:
            result.append(combine_flat_extension(ext, flat_list=image_list, 
                                                 scaling=fscales))
    
    #.colocando as extensoes combinadas na imagem de saida 
    for res in result:
        ext = res[0]
        mflat[ext].data = res[1].data
        mflat[ext].header = res[1].header

    #.salvando imagem de saida
    mflat[0].header['FILENAME']=output
    mflat.writeto(output, overwrite=True)
    mflat.close()

    if delete_images: 
        for file in image_list: 
            try: os.remove(file)
            except OSError as e: print("Error: %s - %s." % (e.filename, e.strerror))


def combine_flat_extension(ext, flat_list, scaling=None):
    
    if scaling is None: fscales = np.full(len(flat_list),1)
    else: fscales = np.array(scaling)
    
    #.agrupando as imagens de flat deste amplificador
    ccd_list = [ CCDData.read(flat_list[j], hdu=ext, unit="adu")
                 for j in np.arange(len(flat_list)) ]
    #.escalonando (manualmente) flats pelo valor mediano
    # (o metodo comb.sigma_clipping nao funciona se a escala for feita pelo comb.scaling)
    for k in np.arange(len(flat_list)): ccd_list[k].data /= fscales[k]
    #.preparando objeto combinador
    comb = Combiner(ccd_list, dtype=np.float32)
    #.processando opcoes do combinador
    comb.sigma_clipping(low_thresh=2., high_thresh=2., func='median', dev_func='mad_std')
    #.realizando combinacao
    comb_flat = comb.median_combine()

    #..pegando o cabecalho da primeira imagem e atualizando
    comb_flat.header = fits.getheader(flat_list[0], ext=ext)
    for n,imgn in enumerate(flat_list, start=1): 
        fstr = os.path.basename(imgn)
        comb_flat.header.append((f"IMCMB{n:03}", f"{fstr}[{ext}] (scale={fscales[n-1]:.1f})"))
    comb_flat.header.append(('NCOMBINE', len(ccd_list), '# images combined'))
    comb_flat.header.append(('FLATNORM', fscales[0], '# normalization scale'))

    #..retornando imagem processada desta extensao
    return (ext, comb_flat)


In [6]:
def process_images(ifc, 
                  master_bias='master_bias.fits',
                  master_flat='master_flat.fits',
                  filter_keywords='filter1,filter2', 
                  multiprocessing=True):
    
    master_flat = master_flat.split('.fits')[0]

    #.identificando 'master flats' presentes na pasta
    mflat_list = np.sort(glob.glob(master_flat+'*.fits'))
    flat_filters = image_filters(mflat_list)

    #.separando imagens de CIENCIA em uma lista 
    object_ifc = ifc.filter(obstype='OBJECT', regex_match=True)
    file_list = object_ifc.files

    #.selecionando o 'master flat' correto para cada imagem
    obj_filters = find_filter(file_list, flat_filters, filter_keywords)

    #.reduzindo imagens de ciencia
    if multiprocessing:
        #..carregando 'master bias' e 'master flat' em um buffer de memoria
        mbias_ccd = image_to_memory(master_bias, name='mbias')
        mflat_ccd = [image_to_memory(flat, name='mflat'+flat.split('.fits')[0][-1]) 
                     for flat in mflat_list]
        #.executando a reducao com multiprocessamento
        with Pool() as pool:
            pool.starmap(partial(reduce_image_mp, master_bias=mbias_ccd,
                         master_flats_list=mflat_ccd), 
                         zip(file_list,obj_filters))
        #..liberando buffers de memoria 
        unlink_memory(mbias_ccd)
        for flat in mflat_ccd: unlink_memory(flat)
    else:
        #.carregando 'master bias' e 'master flat' em objetos CCDDATA
        mbias_ccd = image_to_ccddata(master_bias)
        mflat_ccd = [image_to_ccddata(image) for image in mflat_list]
        #.executando a reducao imagem a imagem
        for i,file in enumerate(file_list):
            reduce_image(file, 
                         master_bias=mbias_ccd, 
                         master_flat=mflat_ccd[obj_filters[i]],
                         merge_amplifiers=True)

    #.atualizando ImageFileCollection
    ifc.refresh()


In [ ]:
#obsolete functions

def overscan_correct(image_file):

    #.Abrindo o arquivo fits da imagem
    hdul = fits.open(image_file)
    image_exts = image_extensions(hdul, is_hdu=True)
    get_date = datetime.now().strftime("%b %d %Y %H:%M")

    #.Loop ao longo das extensoes (amplificadores) de uma imagem
    skip = 0
    for ext in image_exts:
        
        #.Lendo dados desta extensao
        img = CCDData(hdul[ext].data, meta=hdul[ext].header, unit="adu")
        
        #.Abortando caso a keyword 'BIASSEC' nao seja encontrada
        fstr = os.path.basename(image_file)
        if not('BIASSEC' in img.header): 
            print(f".Skipping {fstr:1s}[{ext:1.0f}] - BIASSEC not found")
            skip += 1
            continue
        else:
            print(f".Processing {fstr:1s}[{ext:1.0f}] - Overscan correction")

        biassec = img.header['BIASSEC']        
        img_osub = ccdproc.subtract_overscan(img,
                    fits_section=biassec, 
                    model=None, median=True,
                    add_keyword={'overscan': f"{get_date} Overscan is {biassec}; model=median"})

        #.SOI exception
        #.(keyword TRIMSEC errada no header)
        trimsec = img.header['TRIMSEC']
        if (hdul[0].header['INSTRUME'].find('SOI') >= 0):
            if (i in [1,3]): trimsec = '[29:540,1:2048]'
            else: trimsec = '[28:539,1:2048]'

        img_trim = ccdproc.trim_image(img_osub, fits_section=trimsec,
                    add_keyword={'trim': f"{get_date} Trim is {trimsec}"})
        
        #.Atualizando o header
        imsz = img_trim.shape
        img_trim.header['DATASEC'] = f"[1:{imsz[1]},1:{imsz[0]}]"
        del img_trim.header['BIASSEC']
        del img_trim.header['TRIMSEC']
        del img_trim.header['BZERO']
        del img_trim.header['BSCALE']

        hdul[ext].data = img_trim.data.astype(np.float32)
        hdul[ext].header = img_trim.header
    
    #.Salvando a imagem processada (overwrite)    
    if skip is not len(image_exts): 
        hdul.writeto(image_file, overwrite=True)
    hdul.close()


def bias_correct(image_file, master_bias_ccddata):

    #.Abrindo o arquivo fits da imagem
    hdul = fits.open(image_file)
    image_exts = image_extensions(hdul, is_hdu=True)
    get_date = datetime.now().strftime("%b %d %Y %H:%M")
    
    skip=0
    for ext in image_exts:

        #.Lendo dados desta extensao
        img = CCDData(hdul[ext].data, meta=hdul[ext].header, unit="adu")
        
        #.Abortando caso a keyword 'ZEROCOR' seja encontrada
        fstr = os.path.basename(image_file)
        if ('ZEROCOR' in img.header): 
            print(f".Skipping {fstr:1s}[{ext:1.0f}] - ZEROCOR found")
            skip += 1
            continue
        else:
            print(f".Processing {fstr:1s}[{ext:1.0f}] - Bias correction")

        fstr = os.path.basename(master_bias_ccddata[0].header['FILENAME'])
        img_zcor = ccdproc.subtract_bias(img, master_bias_ccddata[ext],
            add_keyword={'ZEROCOR': f"{get_date} Zero is {fstr}[{ext}]"})

        hdul[ext].data = img_zcor.data.astype(np.float32)
        hdul[ext].header = img_zcor.header
    
    #.Salvando a imagem processada (overwrite)    
    if skip is not len(image_exts): 
        hdul.writeto(image_file, overwrite=True)
    hdul.close()


def flat_correct(image_file, master_flat_ccddata, scale=1):

    #.Abrindo o arquivo fits da imagem
    hdul = fits.open(image_file)
    image_exts = image_extensions(hdul, is_hdu=True)
    get_date = datetime.now().strftime("%b %d %Y %H:%M")
    
    skip=0
    for ext in image_exts:

        #.Lendo dados desta extensao
        img = CCDData(hdul[ext].data, meta=hdul[ext].header, unit="adu")
        
        #.Abortando caso a keyword 'ZEROCOR' seja encontrada
        fstr = os.path.basename(image_file)
        if ('FLATCOR' in img.header): 
            print(f".Skipping {fstr:1s}[{ext:1.0f}] - FLATCOR found")
            skip += 1
            continue
        else:
            print(f".Processing {fstr:1s}[{ext:1.0f}] - Flat correction")

        fstr = os.path.basename(master_flat_ccddata[0].header['FILENAME'])
        fscl = master_flat_ccddata[ext].header['FLATNORM']
        img_fcor = ccdproc.flat_correct(img, master_flat_ccddata[ext], norm_value=scale,
            add_keyword={'FLATCOR': f"{get_date} Flat is {fstr}[{ext}] (norm={fscl:.1f})"})

        hdul[ext].data = img_fcor.data.astype(np.float32)
        hdul[ext].header = img_fcor.header
    
    #.Salvando a imagem processada (overwrite)    
    if skip is not len(image_exts): 
        hdul.writeto(image_file, overwrite=True)
    hdul.close()

    

In [7]:
keys=['obstype', 'time-obs', 'rotoffs', 'ccdsum', 'filter1', 'filter2', 
      'exptime', 'dimmsee', 'airmass', 'object']

folder='./20210710/'

ifc = ImageFileCollection(folder, keywords=keys, ext=0, glob_exclude='*master*.fits', glob_include='*.fits')
tab=ifc.summary
tab.write(folder+'observation.log',format='ascii.fixed_width_two_line',overwrite=True)

process_bias(ifc, combined_bias=folder+'master_bias.fits', multiprocessing=True)

process_flat(ifc, filter_keywords='filter1,filter2', master_bias=folder+'master_bias.fits', 
             combined_flats=folder+'master_flat.fits', multiprocessing=True)

process_images(ifc, filter_keywords='filter1,filter2', master_bias=folder+'master_bias.fits', 
               master_flat=folder+'master_flat.fits', multiprocessing=True)

# current, peak = tracemalloc.get_traced_memory()
# print(f"Current memory usage {current/1e6}MB; Peak: {peak/1e6}MB")
# tracemalloc.stop()

# filters = ifc_filters(ifc, filter_keywords=filter_keywords, obstype_selection='object')
# flat_filters = ifc_filters(ifc, filter_keywords=filter_keywords, obstype_selection='DFLAT|SFLAT|FLAT')

# #.Realizando backup das imagens
# # if not os.path.isdir(backup):
# #     sh.copy2(folder, backup)

# tab=ifc.summary
# tab.write('observation.log',format='ascii.fixed_width_two_line',overwrite=True)

# # if os.path.isdir(folder): os.chdir(folder)
# # else: sys.exit(f"ABORTING: folder {folder} does not exist")

# tab.pprint_all()

.Processing bias2x2.009.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.010.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.011.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.012.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.013.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.014.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.015.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.016.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.017.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.018.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.019.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.020.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.021.fits [1]o--- [2]o--- [3]o--- [4]o---
.Processing bias2x2.022.fits [1]o--- [2]o--- [3]o--- [4]o---


the RADECSYS keyword is deprecated, use RADESYSa. [astropy.wcs.wcs]
the RADECSYS keyword is deprecated, use RADESYSa.
a floating-point value was expected. [astropy.wcs.wcs]
a floating-point value was expected.


.Creating 'master_bias.fits': 14 images
.Processing sflat_B.012.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.013.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.014.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.015.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.016.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.017.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.018.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.019.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.020.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.021.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_B.022.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_Ha.001.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_Ha.002.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_Ha.003.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_Ha.004.fits [1]oz-- [2]oz-- [3]oz-- [4]oz--
.Processing sflat_Ha.005.fits [1]oz-- [2]

the RADECSYS keyword is deprecated, use RADESYSa. [astropy.wcs.wcs]
the RADECSYS keyword is deprecated, use RADESYSa.
a floating-point value was expected. [astropy.wcs.wcs]
a floating-point value was expected.


Creating 'master_flat2.fits': 11 images (s0026 SAM B-k-c-3x3)
Creating 'master_flat3.fits': 10 images (s0027 SAM V-k-c-3x3)
Creating 'master_flat4.fits': 11 images (s0028 SAM R-k-c-3x3)
Creating 'master_flat5.fits': 10 images (s0029 SAM I-k-c-3x3)
.Processing SO2021A-002.055.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.057.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.058.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.059.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.060.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.061.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.062.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.063.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.064.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.065.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.066.fits [1]ozfm [2]ozfm [3]ozfm [4]ozfm
.Processing SO2021A-002.067.fits [1]o

/home/kicage/.miniconda3/envs/dataproc-env/lib/python3.9/site-packages/ccdproc/image_collection.py:544: UserWarning: Header from file "./20210710/SO2021A-002.055.fits" contains multiple entries for "dtm1_1", the pair "dtm1_1=1.0" will be ignored.
  warnings.warn(
/home/kicage/.miniconda3/envs/dataproc-env/lib/python3.9/site-packages/ccdproc/image_collection.py:544: UserWarning: Header from file "./20210710/SO2021A-002.055.fits" contains multiple entries for "dtm2_2", the pair "dtm2_2=1.0" will be ignored.
  warnings.warn(
/home/kicage/.miniconda3/envs/dataproc-env/lib/python3.9/site-packages/ccdproc/image_collection.py:544: UserWarning: Header from file "./20210710/SO2021A-002.055.fits" contains multiple entries for "dtv1", the pair "dtv1=0.0" will be ignored.
  warnings.warn(
/home/kicage/.miniconda3/envs/dataproc-env/lib/python3.9/site-packages/ccdproc/image_collection.py:544: UserWarning: Header from file "./20210710/SO2021A-002.055.fits" contains multiple entries for "dtv2", the pa

In [ ]:
process_bias(ifc, combined_bias=folder+'master_bias.fits', multiprocessing=True)

In [ ]:
process_flat(ifc, filter_keywords='filter1,filter2', master_bias=folder+'master_bias.fits', 
             combined_flats=folder+'master_flat.fits', multiprocessing=True)

In [ ]:
process_images(ifc, filter_keywords='filter1,filter2', master_bias=folder+'master_bias.fits', 
               master_flat=folder+'master_flat.fits', multiprocessing=True)


In [ ]:
#.Initializing main function from the command line
if __name__ == '__main__':

    #..mapping command line arguments to function arguments.
    parser = ArgumentParser(
        prog="data_reduce", 
        description="Main script to perform data reduction of imaging exposures"
    )
    parser.add_argument(
        "folder", type=str,
        help="folder containing the science and calibration images to reduce"
    )
    parser.add_argument(
        "--filter_keywords", nargs="+", default=['filter'],
        help="header keywords containg the photometric filters used (default: %(default)s)"
    )
    parser.add_argument(
        "--summary_file", type=str, default="observations.log",
        help="output table: summary of main header keywords of each image (default: %(default)s)"
    )
    parser.add_argument(
        "--keywords", nargs="+", action="append",
        default=['obstype','time-obs','ccdsum','airmass','exptime','object'],
        help="keywords used to build the 'summary_file' output table (default: %(default)s)"
    )
    parser.add_argument(
        "--logfile", type=str, default="data_reduce.log",
        help="file to log the opperations performed over the images (default: %(default)s)"
    )
    parser.keywords += parser.filter_keywords

    args=parser.parse_args()

    print(args)